In [ ]:
import cv2
import numpy as np
import os

In [ ]:
# --- Configuration ---
VIDEO_PATH  = "Input/video.mp4"   # path to your input video
OUTPUT_DIR  = "Output/matrices"   # where .npy matrix files will be saved
FRAME_STEP  = 1                   # save every Nth frame (1 = every frame)
GRAYSCALE   = False               # True  → (H, W)    matrix per frame
                                  # False → (H, W, 3) matrix per frame (BGR)

In [ ]:
def extract_frames(video_path: str) -> list[np.ndarray]:
    """Read every FRAME_STEP-th frame from the video and return as a list of numpy arrays."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps     = cap.get(cv2.CAP_PROP_FPS)
    width   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Video: {total} frames @ {fps:.1f} fps  |  {width}x{height}")

    frames = []
    idx    = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % FRAME_STEP == 0:
            if GRAYSCALE:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            frames.append(frame)
        idx += 1

    cap.release()
    print(f"Extracted {len(frames)} frames (every {FRAME_STEP} frame(s))")
    return frames

In [ ]:
def save_matrices(frames: list[np.ndarray], output_dir: str):
    """
    Save each frame as a uint8 numpy matrix (.npy).
    Shape is (H, W) for grayscale or (H, W, 3) for color.
    Your HLS Gaussian blur block can load these with np.load().
    """
    os.makedirs(output_dir, exist_ok=True)

    for i, frame in enumerate(frames):
        out_path = os.path.join(output_dir, f"frame_{i:05d}.npy")
        np.save(out_path, frame)

    print(f"Saved {len(frames)} matrices to '{output_dir}'")
    print(f"Matrix shape: {frames[0].shape}  dtype: {frames[0].dtype}")

In [ ]:
def load_matrix(path: str) -> np.ndarray:
    """Helper to load a saved matrix back from disk."""
    return np.load(path)

In [ ]:
# --- Run ---
frames = extract_frames(VIDEO_PATH)
save_matrices(frames, OUTPUT_DIR)

In [ ]:
# --- Quick sanity check ---
# Load the first saved matrix and verify it looks correct
sample = load_matrix(os.path.join(OUTPUT_DIR, "frame_00000.npy"))
print(f"Loaded matrix  shape: {sample.shape}")
print(f"               dtype: {sample.dtype}")
print(f"               min/max: {sample.min()} / {sample.max()}")